# Biohub inference (Kaggle submit)

Detect → graph upgrade → `/kaggle/working/submission.csv`.
Columns: `id,dataset,row_type,node_id,t,z,y,x,source_id,target_id`.

Attach **GPU** notebook inputs. Weight datasets match the production `.957` notebook:

| Role | Kaggle dataset |
| --- | --- |
| Code (`src/`, `configs/`) | `l1ghtsource/biohub` |
| Wheels | `pilkwang/biohub-tracking-support-pack-50ep-v1` (`wheels/`) |
| P1 | `pilkwang/biohub-tracking-support-pack-50ep-v1` |
| P2 | `pilkwang/biohub-temporal-unet3d-seed314159-v1` |
| Model C | `tweakai/biohub-model-c-combined-primary-v1` |
| Cardinality | `tweakai/biohub-public934-source-cardinality-v2` |
| UniGRAFT P1/P2 | `tweakai/biohub-unigraft-p1p2-independent-v1` |
| Live V2 | `tweakai/biohub-live-v2-global-bundle-v2` |
| Ownership | `tweakai/biohub-ownership-exact-v2` |
| EdgeGRAFT | `tweakai/biohub-edgegraft-v3-full-population-v1` |
| CandidateGRAFT | `tweakai/biohub-candidategraft-direct-v1` |
| Motion | `tweakai/biohub-motion-corrector-v1` |
| DeepCenter | `pilkwang/biohub-deepcenter-unet3d-center-prior-v1` |


In [ ]:
import importlib
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

INPUT_DIR = Path('/kaggle/input')
DATASETS_DIR = INPUT_DIR / 'datasets'
WORKING_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
ON_KAGGLE = Path('/kaggle/input').is_dir()
COMPETITION = 'biohub-cell-tracking-during-development'
ALLOW_PIP_INSTALL = False

CODE = {'dataset': 'l1ghtsource/biohub'}
WHEELS = {
    'dataset': 'pilkwang/biohub-tracking-support-pack-50ep-v1',
    'dir': 'wheels',
}


def dataset_root(dataset: str) -> Path:
    if '/' in dataset:
        owner, name = dataset.split('/', 1)
        candidates = (
            DATASETS_DIR / owner / name,
            INPUT_DIR / name,
            INPUT_DIR / owner / name,
        )
    else:
        candidates = (INPUT_DIR / dataset, DATASETS_DIR / dataset)
    for path in candidates:
        if path.exists():
            return path.resolve()
    raise FileNotFoundError(f'Attach Kaggle dataset {dataset}')


if not ON_KAGGLE:
    here = Path.cwd().resolve()
    repo = next(
        candidate
        for candidate in (here, *here.parents)
        if (candidate / 'src' / 'biohub').is_dir()
    )
    INPUT_DIR = repo / 'kaggle' / 'input'
    DATASETS_DIR = INPUT_DIR / 'datasets'
    WORKING_DIR = repo / 'runs' / 'notebook_infer'
    WORKING_DIR.mkdir(parents=True, exist_ok=True)

PACKAGE_SPECS = {
    'tracksdata': 'tracksdata', 'zarr': 'zarr>=3.0.10,<4', 'pyscipopt': 'pyscipopt',
    'geff': 'geff>=1.1.3.1.1', 'geff_spec': 'geff-spec<1.2', 'ilpy': 'ilpy>=0.5.1',
    'polars': 'polars>=1.36', 'blosc2': 'blosc2', 'dask': 'dask',
    'imagecodecs': 'imagecodecs', 'skimage': 'scikit-image>=0.24', 'pyarrow': 'pyarrow',
    'rustworkx': 'rustworkx>=0.17.1', 'sqlalchemy': 'sqlalchemy>=2',
    'numcodecs': 'numcodecs>=0.13,<0.16', 'donfig': 'donfig>=0.8',
    'google_crc32c': 'google-crc32c>=1.5', 'bidict': 'bidict>=0.23.1',
    'psygnal': 'psygnal>=0.14', 'rich': 'rich', 'networkx': 'networkx>=3.2.1',
    'pydantic': 'pydantic>=2.11', 'pydantic_core': 'pydantic-core',
    'annotated_types': 'annotated-types', 'typing_extensions': 'typing-extensions>=4.13',
    'typing_inspection': 'typing-inspection', 'markdown_it': 'markdown-it-py',
    'pygments': 'pygments', 'click': 'click', 'cloudpickle': 'cloudpickle',
    'fsspec': 'fsspec', 'partd': 'partd', 'locket': 'locket', 'toolz': 'toolz',
    'yaml': 'pyyaml', 'ndindex': 'ndindex', 'msgpack': 'msgpack', 'numexpr': 'numexpr',
    'deprecated': 'deprecated', 'wrapt': 'wrapt', 'imageio': 'imageio', 'PIL': 'pillow',
    'tifffile': 'tifffile', 'lazy_loader': 'lazy-loader', 'tqdm': 'tqdm',
    'threadpoolctl': 'threadpoolctl',
}
EXTRA_SPECS = {
    'tracksdata': ['bidict>=0.23.1', 'psygnal>=0.14', 'rich'],
    'zarr': ['donfig>=0.8', 'google-crc32c>=1.5', 'numcodecs>=0.13,<0.16'],
    'geff': ['geff-spec<1.2', 'networkx>=3.2.1', 'pydantic>=2.11', 'numcodecs>=0.13,<0.16'],
    'geff_spec': ['pydantic>=2.11', 'annotated-types', 'pydantic-core', 'typing-inspection'],
    'polars': ['polars-runtime-32'],
    'dask': ['click', 'cloudpickle', 'fsspec', 'partd', 'pyyaml', 'toolz'],
    'partd': ['locket'], 'blosc2': ['ndindex', 'msgpack', 'numexpr'],
    'numcodecs': ['deprecated', 'msgpack', 'wrapt'], 'rich': ['markdown-it-py', 'pygments'],
    'pydantic': ['annotated-types', 'pydantic-core', 'typing-extensions>=4.13',
                 'typing-inspection'],
    'skimage': ['imageio', 'pillow', 'tifffile', 'lazy-loader', 'networkx'],
}
os.environ.setdefault('POLARS_PREFER_PKG', '32')


def _wheel_dirs() -> list[Path]:
    found = []
    try:
        hinted = dataset_root(WHEELS['dataset']) / WHEELS['dir']
        if hinted.is_dir() and any(hinted.glob('*.whl')):
            found.append(hinted)
    except FileNotFoundError:
        pass
    if DATASETS_DIR.is_dir():
        for wheels in DATASETS_DIR.glob('*/*/wheels'):
            if wheels.is_dir() and any(wheels.glob('*.whl')) and wheels not in found:
                found.append(wheels)
    print('wheel dirs:', found, flush=True)
    return found


def _importable(name: str) -> bool:
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


def _runtime_healthy() -> list[str]:
    broken = []
    try:
        import polars as pl
        from polars._plr import PySeries  # noqa: F401

        if not hasattr(pl, 'Float16'):
            broken.append('polars')
    except Exception:
        broken.append('polars')
    try:
        import zarr

        if int(str(zarr.__version__).split('.', 1)[0]) < 3:
            broken.append('zarr')
    except Exception:
        broken.append('zarr')
    return broken


def _pip(names: list[str], *, force_reinstall: bool) -> None:
    specs, seen = [], set()
    for name in names:
        for spec in [PACKAGE_SPECS.get(name, name), *EXTRA_SPECS.get(name, [])]:
            if spec.lower() not in seen:
                seen.add(spec.lower())
                specs.append(spec)
    command = [sys.executable, '-m', 'pip', 'install', '--no-deps']
    if force_reinstall:
        command.append('--force-reinstall')
    wheels = _wheel_dirs()
    if not wheels:
        raise ImportError(f'No wheels/ with *.whl. Attach {WHEELS["dataset"]}.')
    offline = [*command, '--no-index']
    for directory in wheels:
        offline += ['--find-links', str(directory)]
    print('pip install', specs, 'force' if force_reinstall else 'missing', flush=True)
    result = subprocess.run([*offline, *specs], text=True)
    if result.returncode == 0:
        return
    if ALLOW_PIP_INSTALL:
        result = subprocess.run([*command, *specs], text=True)
        if result.returncode == 0:
            return
    raise ImportError(f'Could not install: {names}. Attach the wheels artifact.')


def _install(names: list[str]) -> None:
    force = [name for name in names if name in {'polars', 'zarr'}]
    rest = [name for name in names if name not in {'polars', 'zarr'}]
    if force:
        _pip(force, force_reinstall=True)
    if rest:
        _pip(rest, force_reinstall=False)


def ensure_dependencies() -> None:
    for round_id in range(5):
        broken = [name for name in PACKAGE_SPECS if not _importable(name)]
        broken += [name for name in _runtime_healthy() if name not in broken]
        print(f'deps round {round_id}: missing {broken}', flush=True)
        if not broken:
            return
        _install(broken)
        for name in list(sys.modules):
            if name.split('.')[0] in {n.split('.')[0] for n in broken}:
                sys.modules.pop(name, None)
    raise ImportError('Dependency recovery did not converge.')


if ON_KAGGLE:
    ensure_dependencies()
    import polars as pl

    print('polars', pl.__version__, 'Float16', hasattr(pl, 'Float16'), flush=True)

if ON_KAGGLE:
    REPO = dataset_root(CODE['dataset'])
    if not (REPO / 'src' / 'biohub').is_dir():
        REPO = next(path for path in REPO.iterdir() if (path / 'src' / 'biohub').is_dir())
else:
    REPO = next(
        candidate
        for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (candidate / 'src' / 'biohub').is_dir()
    )

os.environ['BIOHUB_ROOT'] = str(REPO)
SRC = str(REPO / 'src')
os.environ['PYTHONPATH'] = SRC + (
    os.pathsep + os.environ['PYTHONPATH'] if os.environ.get('PYTHONPATH') else ''
)
if SRC not in sys.path:
    sys.path.insert(0, SRC)

print('ON_KAGGLE', ON_KAGGLE)
print('REPO', REPO)
print('INPUT_DIR', INPUT_DIR)
print('DATASETS_DIR', DATASETS_DIR)
print('WORKING_DIR', WORKING_DIR)


In [ ]:
import shutil
import zipfile

COMP_DIR = next(
    (
        path
        for path in (INPUT_DIR / 'competitions' / COMPETITION, INPUT_DIR / COMPETITION)
        if path.exists()
    ),
    INPUT_DIR / 'competitions' / COMPETITION,
)
LOCAL_BUNDLE = DATASETS_DIR / 'antonoof' / 'all_files'

MODELS = {
    'p1': {
        'dataset': 'pilkwang/biohub-tracking-support-pack-50ep-v1',
        'kaggle': 'weights/unet_transformer/split_0/edge_predictor_best.pth',
        'local': 'p1/weights/unet_transformer/split_0/edge_predictor_best.pth',
    },
    'p2': {
        'dataset': 'pilkwang/biohub-temporal-unet3d-seed314159-v1',
        'kaggle': 'unet_transformer/split_0/edge_predictor_best.pth',
        'local': 'p2/weights/unet_transformer/split_0/edge_predictor_best.pth',
    },
    'model_c_dir': {
        'dataset': 'tweakai/biohub-model-c-combined-primary-v1',
        'kaggle': '',
        'local': 'model_c',
    },
    'model_c': {
        'dataset': 'tweakai/biohub-model-c-combined-primary-v1',
        'kaggle': 'model_c/edge_predictor_best.pth',
        'local': 'model_c/weights/edge_predictor_best.pth',
    },
    'source_cardinality': {
        'dataset': 'tweakai/biohub-public934-source-cardinality-v2',
        'kaggle': '',
        'local': 'source_cardinality',
    },
    'unigraft_p1p2': {
        'dataset': 'tweakai/biohub-unigraft-p1p2-independent-v1',
        'kaggle': '',
        'local': 'unigraft_p1p2',
    },
    'live_v2_global_bundle': {
        'dataset': 'tweakai/biohub-live-v2-global-bundle-v2',
        'kaggle': '',
        'local': 'live_v2_bundle',
    },
    'ownership_full_population': {
        'dataset': 'tweakai/biohub-ownership-exact-v2',
        'kaggle': '',
        'local': 'ownership',
    },
    'edgegraft': {
        'dataset': 'tweakai/biohub-edgegraft-v3-full-population-v1',
        'kaggle': '',
        'local': 'edgegraft',
    },
    'candidategraft': {
        'dataset': 'tweakai/biohub-candidategraft-direct-v1',
        'kaggle': '',
        'local': 'candidategraft',
    },
    'motion_corrector': {
        'dataset': 'tweakai/biohub-motion-corrector-v1',
        'kaggle': 'motion_corrector_best.pt',
        'local': 'motion_corrector/motion_corrector_best.pt',
    },
    'deepcenter': {
        'dataset': 'pilkwang/biohub-deepcenter-unet3d-center-prior-v1',
        'kaggle': 'weights/full_frame_center/best.pt',
        'local': 'deepcenter/best.pt',
    },
}


def _unpack_weights(root: Path) -> Path:
    destination = WORKING_DIR / f'{root.name}_weights'
    ready = destination / 'unet_transformer' / 'split_0' / 'edge_predictor_best.pth'
    if ready.is_file():
        return destination
    weights_dir = root / 'weights'
    nested = weights_dir / 'unet_transformer' / 'split_0' / 'edge_predictor_best.pth'
    if nested.is_file():
        return weights_dir
    weights_zip = root / 'weights.zip'
    if destination.exists():
        shutil.rmtree(destination)
    if weights_dir.is_dir():
        shutil.copytree(weights_dir, destination)
        return destination
    if weights_zip.is_file():
        destination.mkdir(parents=True)
        with zipfile.ZipFile(weights_zip) as archive:
            archive.extractall(destination)
        return destination
    raise FileNotFoundError(f'Missing weights/ and weights.zip in {root}')


def _resolve_file(root: Path, rel: str) -> Path:
    candidates = [root / rel]
    if rel.startswith('weights/'):
        candidates.append(root / rel[len('weights/') :])
    else:
        candidates.append(root / 'weights' / rel)
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    can_unpack = (root / 'weights.zip').is_file() or (
        root / 'weights' / 'unet_transformer' / 'split_0' / 'edge_predictor_best.pth'
    ).is_file()
    if can_unpack:
        unpacked = _unpack_weights(root)
        packed_rel = rel[len('weights/') :] if rel.startswith('weights/') else rel
        for candidate in (unpacked / packed_rel, unpacked / 'weights' / packed_rel, unpacked / rel):
            if candidate.is_file():
                return candidate.resolve()
        matches = sorted(
            {path.resolve() for path in unpacked.rglob(Path(rel).name) if path.is_file()}
        )
        if len(matches) == 1:
            return matches[0]
    matches = sorted({path.resolve() for path in root.rglob(Path(rel).name) if path.is_file()})
    if len(matches) == 1:
        return matches[0]
    raise FileNotFoundError(f'{rel} not found under {root}')


def artifact(spec: dict, marker: str = '') -> Path:
    if ON_KAGGLE:
        root = dataset_root(spec['dataset'])
        rel = spec['kaggle']
        path = (root / rel) if rel else root
    else:
        path = LOCAL_BUNDLE / spec['local']
    path = path.resolve()
    if not marker:
        if path.is_file() or not ON_KAGGLE:
            return path
        return _resolve_file(dataset_root(spec['dataset']), spec['kaggle'])
    if (path / marker).is_file():
        return path
    parent = path.parent
    if path != parent and (parent / marker).is_file():
        return parent.resolve()
    if path.is_dir():
        matches = sorted(
            child.resolve()
            for child in path.iterdir()
            if child.is_dir() and (child / marker).is_file()
        )
        if len(matches) == 1:
            return matches[0]
    return path


bundle_dir = dataset_root(MODELS['p1']['dataset']) if ON_KAGGLE else LOCAL_BUNDLE
bundle_paths = {
    'bundle_dir': bundle_dir,
    'p1_checkpoint': artifact(MODELS['p1']),
    'p2_checkpoint': artifact(MODELS['p2']),
    'model_c_dir': artifact(MODELS['model_c_dir'], 'deploy_spec.json'),
    'model_c_checkpoint': artifact(MODELS['model_c']),
    'source_cardinality_dir': artifact(
        MODELS['source_cardinality'], 'source_cardinality_graph_runtime.py'
    ),
    'unigraft_p1p2_dir': artifact(
        MODELS['unigraft_p1p2'], 'p1p2_only_source_cardinality_head.pt'
    ),
    'live_v2_bundle_dir': artifact(
        MODELS['live_v2_global_bundle'], 'live_v2_global_bundle_runtime_v1.py'
    ),
    'ownership_dir': artifact(MODELS['ownership_full_population'], 'deploy_spec.json'),
    'edgegraft_dir': artifact(MODELS['edgegraft'], 'edgegraft_v3_deploy_runtime_v5.py'),
    'candidategraft_dir': artifact(
        MODELS['candidategraft'], 'candidategraft_direct_runtime_v1.py'
    ),
    'motion_checkpoint': artifact(MODELS['motion_corrector']),
    'deepcenter_checkpoint': artifact(MODELS['deepcenter']),
}

test_dir = (COMP_DIR / 'test') if ON_KAGGLE else (COMP_DIR / 'train')
train_dir = COMP_DIR / 'train' if (COMP_DIR / 'train').is_dir() else test_dir
config_path = REPO / 'configs' / ('infer_kaggle.yaml' if ON_KAGGLE else 'infer.yaml')
movie_ids = (
    sorted(path.name.removesuffix('.zarr') for path in test_dir.glob('*.zarr'))
    if ON_KAGGLE
    else ['44b6_0113de3b']
)
if not movie_ids:
    raise FileNotFoundError(f'No *.zarr movies in {test_dir}')

print('COMP_DIR', COMP_DIR, 'exists', COMP_DIR.exists())
print('test_dir', test_dir)
print('p1', bundle_paths['p1_checkpoint'], 'exists', bundle_paths['p1_checkpoint'].is_file())
print('p2', bundle_paths['p2_checkpoint'], 'exists', bundle_paths['p2_checkpoint'].is_file())
print(
    'model_c_dir',
    bundle_paths['model_c_dir'],
    'deploy_spec',
    (bundle_paths['model_c_dir'] / 'deploy_spec.json').is_file(),
)
print('movies', len(movie_ids), movie_ids[:8])
missing = [
    f'{key}={path}'
    for key, path in bundle_paths.items()
    if str(key).endswith('_checkpoint') and not Path(path).is_file()
]
if missing:
    raise FileNotFoundError('Missing checkpoints: ' + '; '.join(missing))


In [ ]:
from biohub.infer.run import assemble_submission, build_tracking, detect_job, run, run_upgrade
from biohub.modules.detect import predict_from_job
from biohub.modules.graph.upgrade import GraphUpgrade

run_dir = WORKING_DIR
work_dir = run_dir / 'workdir'
work_dir.mkdir(parents=True, exist_ok=True)
tracking = build_tracking(
    config_path=config_path,
    bundle_dir=bundle_dir,
    test_dir=test_dir,
    work_dir=work_dir,
    bundle_paths=bundle_paths,
)
tracking.paths.test_dir, tracking.bundle.p1_checkpoint


In [ ]:
from biohub.modules.detect import predict_from_job

kaggle_submission = (
    Path('/kaggle/working') / 'submission.csv' if ON_KAGGLE else work_dir / 'submission.csv'
)

if ON_KAGGLE:
    summary = run(
        movie_ids=movie_ids,
        bundle_dir=bundle_dir,
        test_dir=test_dir,
        train_dir=train_dir,
        run_dir=run_dir,
        gpu_count=tracking.runtime.gpu_workers,
        config=tracking,
    )
    produced = Path(summary['submission'])
    if produced.resolve() != kaggle_submission.resolve():
        kaggle_submission.write_bytes(produced.read_bytes())
    stats = summary
else:
    upgrade = GraphUpgrade(tracking)
    run_upgrade(upgrade)
    upgrade.csv_columns
    job = detect_job(
        movie_ids=movie_ids,
        test_dir=tracking.paths.test_dir,
        predictions_dir=work_dir / 'predictions',
        tracking=tracking,
    )
    pred_dir = predict_from_job(job)
    geff_path = pred_dir / f'{movie_ids[0]}.geff'
    stats = upgrade.process_graph(geff_path, 'combined')
    node_rows, edge_rows = assemble_submission(
        movie_ids,
        tracking.paths.shard_dir,
        kaggle_submission,
        upgrade.csv_columns,
    )
    stats = {'nodes': node_rows, 'edges': edge_rows, **stats}

print('submission', kaggle_submission, 'exists', kaggle_submission.is_file())
stats


In [ ]:
import polars as pl

from biohub.infer.run import assemble_submission

preview = pl.read_csv(kaggle_submission)
{
    'path': str(kaggle_submission),
    'columns': list(preview.columns),
    'n_rows': preview.height,
    'datasets': preview['dataset'].n_unique(),
    'preview': preview.head(),
}
